# BLS Data Preparation

This Notebook is meant to prepare the data from the dumps for Bureau of Labor Statistics for the GHP POC.

# Setup

## Constants

In [6]:
PROJECT_ID = "ghp-poc"

LBS_BUCKET_NAME = "bls_ghp_poc"

## Imports

In [32]:
import io
from typing import Any, Dict, List

import pandas as pd

from google.cloud import bigquery
from google.cloud import storage

# Helper functions

In [5]:
def read_csv(bucket_name: str, blob_name: str):
    """Download CSV file from GCS."""
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_name)
    csv_string = blob.download_as_string().decode('utf-8')
    df = pd.read_csv(io.StringIO(csv_string))
    return df

In [44]:
def read_txt_file(bucket_name: str, blob_name: str):
    """Get text contents from GCS txt file."""
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_name)

    file_content = blob.download_as_text()
    return file_content


In [33]:
def load_or_append_to_bq(
    project_id: str,
    dataset_id: str,
    table_id: str,
    data: List[Dict[str, Any]],
    schema: List[bigquery.SchemaField],
    write_disposition: bigquery.WriteDisposition = bigquery.WriteDisposition.WRITE_APPEND, # pylint: disable=line-too-long
) -> None:
    """Loads or appends data to a BigQuery table.
    Args:
        project_id: Google Cloud project ID.
        dataset_id: BigQuery dataset ID.
        table_id: BigQuery table ID.
        data: List of dictionaries representing the data to load.
        schema: BigQuery table schema.
        write_disposition: Write disposition for the load job.
                           Defaults to WRITE_APPEND.
    """

    client = bigquery.Client(project=project_id)
    table_ref = client.dataset(dataset_id).table(table_id)

    # Check if the table exists, create it if not
    try:
        client.get_table(table_ref)
    except Exception: # pylint: disable=broad-except
        table = bigquery.Table(table_ref, schema=schema)
        client.create_table(table)

        # If the table is newly created, force WRITE_TRUNCATE
        write_disposition = bigquery.WriteDisposition.WRITE_TRUNCATE

    job_config = bigquery.LoadJobConfig(
        schema=schema,
        source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
        write_disposition=write_disposition,
    )

    # Load the data into BQ.
    client.load_table_from_json(data, table_ref, job_config=job_config).result()

# Prepare Median Hourly Wage

Source: https://www.bls.gov/oes/tables.htm

## Load CSV from GCS

In [17]:
MEDIAN_WAGES_FILE_BLOB = "bls_median_wages_metro.csv"
media_wages_metro_df = read_csv(
    bucket_name=LBS_BUCKET_NAME,
    blob_name=MEDIAN_WAGES_FILE_BLOB
)

## Process CSV for BQ

In [18]:
# All occupations
all_occupations_median = media_wages_metro_df[media_wages_metro_df["OCC_TITLE"] == "All Occupations"]

In [21]:
all_occupations_median = all_occupations_median[["AREA_TITLE", "H_MEDIAN"]]
# source: https://www.bls.gov/oes/tables.htm

In [22]:
all_occupations_median["source"] = "https://www.bls.gov/oes/tables.htm"

/var/folders/_b/nqbkzf8939538f6tjvldq6j80107pk/T/ipykernel_34888/782568933.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_occupations_median["source"] = "https://www.bls.gov/oes/tables.htm"


In [27]:
all_occupations_median.rename(columns={
    "AREA_TITLE": "metro",
    "H_MEDIAN": "median_hourly_wage"
}, inplace=True)

all_occupations_median["median_hourly_wage"] = all_occupations_median["median_hourly_wage"].astype(float)

/var/folders/_b/nqbkzf8939538f6tjvldq6j80107pk/T/ipykernel_34888/2609753584.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_occupations_median.rename(columns={


## Upload Median Wages to BQ

In [39]:
BLSDATASET_ID = "bls"

MEDIAN_WAGE_TABLE_ID = "metro_median_hourly_wages"

In [40]:
median_data_bq = all_occupations_median.to_dict(orient="records")

# Schema.
median_wage_schema = [
    bigquery.SchemaField("metro", "STRING"),
    bigquery.SchemaField("median_hourly_wage", "STRING"),
    bigquery.SchemaField("source", "STRING")
]

In [41]:
load_or_append_to_bq(
    project_id=PROJECT_ID,
    dataset_id=BLSDATASET_ID,
    table_id=MEDIAN_WAGE_TABLE_ID,
    schema=median_wage_schema,
    data=median_data_bq
)

# Prepare Labor Force Data

Source: https://www.bls.gov/news.release/metro.t01.htm

## Load text file of labor

In [45]:
LABOR_TXT_FILE = "labor_force.txt"
LABOR_TABLE_TEXT = read_txt_file(
    bucket_name=LBS_BUCKET_NAME,
    blob_name=LABOR_TXT_FILE
)

## Process Labor Text File

In [47]:
labor_lines = LABOR_TABLE_TEXT.split("\n")

In [ ]:
states = [
    "Alabama", "Alaska", "Arizona", "Arkansas", "California",
    "Colorado", "Connecticut", "Delaware", "Florida", "Georgia",
    "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa",
    "Kansas", "Kentucky", "Louisiana", "Maine", "Maryland",
    "Massachusetts", "Michigan", "Minnesota", "Mississippi", "Missouri",
    "Montana", "Nebraska", "Nevada", "New Hampshire", "New Jersey",
    "New Mexico", "New York", "North Carolina", "North Dakota", "Ohio",
    "Oklahoma", "Oregon", "Pennsylvania", "Rhode Island", "South Carolina",
    "South Dakota", "Tennessee", "Texas", "Utah", "Vermont",
    "Virginia", "Washington", "West Virginia", "Wisconsin", "Wyoming"
]

In [52]:
labor_data_processed = []
for i in range(0, len(labor_lines), 3):
    labor_chunk = labor_lines[i:i + 3]
    area_name = labor_chunk[0]
    if area_name in states:
        area_type = "state"
    else:
        area_type = "metro"
    civilian_data = labor_chunk[2].split("\t")
    labor_force = civilian_data[3]
    unemployment_rate = civilian_data[-1]
    date = "December 2024"
    source = "https://www.bls.gov/news.release/metro.t01.htm"
    labor_data_processed.append({
        "area_name": area_name,
        "area_type": area_type,
        "labor_force": labor_force,
        "unemployment_rate": unemployment_rate,
        "date": date,
        "source": source
    })

In [54]:
labor_data_processed[0]

{'area_name': 'Alabama',
 'area_type': 'state',
 'labor_force': '2,363,548',
 'unemployment_rate': '3.2',
 'date': 'December 2024',
 'source': 'https://www.bls.gov/news.release/metro.t01.htm'}

## Upload Labor to BQ

In [55]:
LABOR_FORCE_TABLE = "labor_force"

# Schema.
labor_force_schema = [
    bigquery.SchemaField("area_name", "STRING"),
    bigquery.SchemaField("area_type", "STRING"),
    bigquery.SchemaField("labor_force", "STRING"),
    bigquery.SchemaField("unemployment_rate", "STRING"),
    bigquery.SchemaField("date", "STRING"),
    bigquery.SchemaField("source", "STRING"),
]

load_or_append_to_bq(
    project_id=PROJECT_ID,
    dataset_id=BLSDATASET_ID,
    table_id=LABOR_FORCE_TABLE,
    schema=labor_force_schema,
    data=labor_data_processed
)

# Prepare State Tax Data

Source: https://taxfoundation.org/data/all/state/state-corporate-income-tax-rates-brackets/

In [56]:
STATE_TAX_TXT_FILE = "state_tax_2025.txt"
STATE_TAX_TXT = read_txt_file(
    bucket_name=LBS_BUCKET_NAME,
    blob_name=STATE_TAX_TXT_FILE
)

In [60]:
taxes_by_state_and_bracket = STATE_TAX_TXT.split("\n")

In [63]:
taxes_brackets_data_list = []
for row in taxes_by_state_and_bracket:
    row_split = row.split(",")

    state = row_split[0]
    if "%" not in row_split[1]:
        tax_rate = 0.0
    else:
        tax_rate = float(row_split[1].split("%")[0])
    taxes_brackets_data_list.append({
        "state": state,
        "tax_rate": tax_rate,
        "year": 2025,
        "source": "https://taxfoundation.org/data/all/state/state-corporate-income-tax-rates-brackets/"
    })


In [65]:
taxes_brackets_data_list_df = pd.DataFrame(data=taxes_brackets_data_list)

In [68]:
max_rates_idxs = taxes_brackets_data_list_df.groupby("state")["tax_rate"].idxmax()

In [72]:
max_tax_rates_df = taxes_brackets_data_list_df.loc[max_rates_idxs]

## Upload Max Tax Rates to BQ

In [74]:
TAX_RATE_TABLE_ID = "state_tax_rates"

tax_rate_data = max_tax_rates_df.to_dict(orient="records")

In [76]:
tax_rate_data[0]

{'state': 'Alabama',
 'tax_rate': 6.5,
 'year': 2025,
 'source': 'https://taxfoundation.org/data/all/state/state-corporate-income-tax-rates-brackets/'}

In [77]:
tax_rate_schema = [
    bigquery.SchemaField("state", "STRING"),
    bigquery.SchemaField("tax_rate", "FLOAT"),
    bigquery.SchemaField("year", "INTEGER"),
    bigquery.SchemaField("source", "STRING")
]

In [78]:
load_or_append_to_bq(
    project_id=PROJECT_ID,
    dataset_id=BLSDATASET_ID,
    table_id=TAX_RATE_TABLE_ID,
    schema=tax_rate_schema,
    data=tax_rate_data
)